In [14]:
from locale import normalize
from unicodedata import numeric

from anyio.functools import P
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import pandas as pd

df = pd.read_csv('../data/raw/Telco_Churn.csv')
df.columns = df.columns.str.strip()

# print(df.shape)
# print(df.columns.tolist())

target = "Churn Value"

drop_cols = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude",
    "Churn Label",
    "Churn Score",
    "CLTV",
    "Churn Reason"
]

selected_features = [
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Tenure Months",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method",
    "Monthly Charges",
    "Total Charges"
]


model_df = df.drop(columns=drop_cols).copy()


model_df["Total Charges"] = pd.to_numeric(model_df["Total Charges"], errors='coerce')

# print(model_df["Total Charges"].isna().sum())

model_df = model_df[selected_features + [target]].copy()

# print(model_df.isna().sum().sort_values(ascending=False))

x = model_df.drop(columns=[target])
y = model_df[target]

# print(x.shape)
# print(y.shape)
# print(y.value_counts())

numeric_features = x.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = x.select_dtypes(include=['object',"string"]).columns.tolist()

# print("Numeric features:", numeric_features)
# print("Categorical features:", categorical_features)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# print(x_train.shape,x_test.shape)
# print(y_train.value_counts(normalize = True))
# print(y_test.value_counts(normalize = True))


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[  
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]   
)

x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

print("processed x_train shape:", x_train_processed.shape)
print("processed x_test shape:", x_test_processed.shape)

model_df.to_csv('../data/processed/model_data_results.csv', index=False)



processed x_train shape: (5634, 46)
processed x_test shape: (1409, 46)
